Great! Here's how you can **integrate `SoftDTW` loss** into your `train_tgan_sequential` function — with **minimal changes** and **full working code snippet**.

---

## ✅ Step-by-step: Add `SoftDTW` loss

### 1. **Install SoftDTW**

If you haven't installed it yet:

```bash
pip install soft-dtw-cuda
```

Or for CPU-only (slower):

```bash
pip install pytorch-softdtw
```

We'll use `soft-dtw-cuda` here for performance.

---

### 2. **Import and Initialize SoftDTW**

Place this at the **top of your script**:

```python
from soft_dtw_cuda import SoftDTW
```

Then, inside your function `train_tgan_sequential(...)`, add this **right after `loss_bce = nn.BCEWithLogitsLoss()`**:

```python
dtw_loss_fn = SoftDTW(gamma=1.0, normalize=True)
```

---

### 3. **Add DTW Loss to Generator Step**

Inside this block:

```python
for iteration in range(gen_iter):
```

After computing `fake_feat_seq` and `real_feat_seq`, insert the DTW loss:

```python
# Add DTW loss (between real and fake feature sequences)
dtw_loss = 0.0
for b in range(B):
    # Inputs must be [T, F]
    fake_seq_b = fake_feat_seq[b]
    real_seq_b = real_feat_seq[b]
    dtw_loss += dtw_loss_fn(fake_seq_b, real_seq_b)
dtw_loss = dtw_loss / B  # Mean over batch
```

---

### 4. **Add `w_dtw` Weight Parameter**

Modify your function signature to accept a new weight:

```python
def train_tgan_sequential(..., w_dtw=0.1):  # Add w_dtw
```

Or hardcode it inside for now:

```python
w_dtw = 0.1  # DTW loss weight
```

---

### 5. **Update Total Generator Loss**

Modify this line:

```python
g_loss = (w_adv_spatial * adv_spatial
          + w_adv_temporal * adv_temporal
          + w_featmatch * feat_match
          + w_recon * recon
          + w_smooth * tv_t)
```

to include `dtw_loss`:

```python
g_loss = (w_adv_spatial * adv_spatial
          + w_adv_temporal * adv_temporal
          + w_featmatch * feat_match
          + w_recon * recon
          + w_smooth * tv_t
          + w_dtw * dtw_loss)  # New!
```

---

### ✅ Final Snippet (Drop-in Code)

Here’s the **drop-in DTW part** to insert inside your generator training loop:

```python
# ---- DTW Loss ----
dtw_loss = 0.0
for b in range(B):
    fake_seq_b = fake_feat_seq[b]  # shape: [T, F]
    real_seq_b = real_feat_seq[b]  # shape: [T, F]
    dtw_loss += dtw_loss_fn(fake_seq_b, real_seq_b)
dtw_loss = dtw_loss / B
```

Then modify `g_loss` to include:

```python
+ w_dtw * dtw_loss
```

---

### 🧪 Tip for Debugging

You can print or log the DTW loss:

```python
print(f"DTW loss: {dtw_loss.item():.4f}")
```

---

## ✅ Summary

* ✅ **SoftDTW** is now integrated
* ✅ Differentiable & GPU-accelerated
* ✅ Adds temporal alignment to feature space
* ✅ Only small changes to your code

---

Would you like help with **SoftDTW installation** or a fallback for **CPU-only mode**?


Thanks for sharing your **normalized `doy_list`** — this clears things up nicely.

---

## 🔍 Quick Analysis of Your Time Gaps

Your `doy_list`:

```python
[0.0246, 0.0658, 0.1014, 0.1425, 0.1699, 0.2110, 0.2301, 0.2712, 
 0.2986, 0.3260, 0.3534, 0.4082, 0.4356, 0.5041, 0.5315, 0.5589, 
 0.5808, 0.6000, 0.6274, 0.6411, 0.6767, 0.7178, 0.7452, 0.7643, 
 0.7863, 0.8329, 0.9288]
```

The differences (i.e., time gaps between frames) are mostly **small**:

```python
diffs = [~0.04, ~0.03, ~0.04, ~0.03, ..., ~0.1, ~0.09]
```

So:

* Your **temporal resolution is high**
* The **TV loss** is **more sensitive**, because the generator shouldn't make huge jumps across short time spans
* Your idea of adjusting `tv_t` using these actual **time gaps** still **applies and is beneficial**

---

## ✅ Recommended Fix: Time-Weighted `tv_t` for Small Gaps

Even if gaps are small, **you still should normalize temporal differences by the time deltas** — so jumps are only penalized when **they're too big for the time gap**.

This makes your model **aware of time flow**, not just index positions.

---

## ✅ Drop-in Code (Updated for Small Gaps)

```python
# Precompute time deltas from normalized doy list (only once per epoch)
doy_tensor = torch.tensor(doy_list, dtype=torch.float32, device=DEVICE)  # shape: [T]
time_deltas = doy_tensor[1:] - doy_tensor[:-1]  # shape: [T-1]

# Avoid division by zero and overly large weights
epsilon = 1e-6
weights = 1.0 / (time_deltas + epsilon)  # shape: [T-1]
weights = torch.clamp(weights, max=100.0)  # optional, avoids huge penalties

# Reshape for broadcasting to [B, T-1, 1, 1, 1]
weights = weights.view(1, -1, 1, 1, 1)

# Compute temporal differences in generated frames
# fake_seq: [B, T, C, H, W]
delta_frames = torch.abs(fake_seq[:, 1:] - fake_seq[:, :-1])  # [B, T-1, C, H, W]

# Apply time-weighted smoothness
tv_t = torch.mean(weights * delta_frames)  # scalar
```

This version:

* Penalizes large jumps **per unit of time**
* Encourages the generator to **model temporal dynamics** more realistically
* Still works well with your **small, irregular gaps**

---

## ⚠️ Tuning Tip

If your `tv_t` loss becomes **too strong** (because of the small gaps and inverse weighting), consider:

* Scaling `w_smooth` down
* Using `torch.clamp` to cap `weights` (as in the example)
* Using `torch.sqrt(1.0 / (delta + epsilon))` instead of `1.0 / delta` for **softer weighting**

---

## ✅ Summary

| ✅ You Have                   | ✔️ Fix                                         |
| ---------------------------- | ---------------------------------------------- |
| Non-uniform, small time gaps | Use **inverse time gap weighting** for `tv_t`  |
| Sensitive generator          | Normalize temporal diffs by `doy_list` gaps    |
| Stable GAN setup             | Use `clamp` and `epsilon` to avoid instability |

---

Would you like to encapsulate this into a helper function like `compute_time_weighted_tv_loss(fake_seq, doy_list)` to keep the training code clean?


In [ ]:
from soft_dtw_cuda import SoftDTW

def train_tgan_sequential(Gs, 
                          Ds, 
                          Gt, 
                          Dt, 
                          g_optimizer, 
                          d_optimizer, 
                          d_optimizer_t,
                          dataloader,
                          doy_list, 
                          epochs, 
                          num_classes=10,
                          modelid=1, 
                          gen_iter=1,
                          w_adv_spatial=1.0,
                          w_adv_temporal=0.5,
                          w_featmatch=0.1, 
                          w_recon=25.0, 
                          w_smooth=2.0,
                          w_dtw=0.1,
                          freeze_spatial_epochs = 0
                          ):
    T = 27

    loss_bce = nn.BCEWithLogitsLoss()
    dtw_loss_fn = SoftDTW(gamma=1.0, normalize=True)  # Differentiable DTW
    losses_g, losses_d = [], []

    # Time-aware smoothing weights from normalized doy_list
    doy_tensor = torch.tensor(doy_list, dtype=torch.float32, device=DEVICE)
    time_deltas = doy_tensor[1:] - doy_tensor[:-1]  # [T-1]
    epsilon = 1e-6
    weights_tv = 1.0 / (time_deltas + epsilon)
    weights_tv = torch.clamp(weights_tv, max=100.0).view(1, -1, 1, 1, 1)  # for broadcasting

    # Generator wrapper
    G = SequentialGeneratorWrapper(Gs, Gt).to(DEVICE)

    for epoch in range(epochs): 
        if epoch == 0:
            print("Saving initial models")
            save_models(G.fetch_spatial_generator(), 
                        G.fetch_temporal_generator(),
                        Ds, Dt, g_optimizer, d_optimizer, d_optimizer_t,
                        epoch, modelid=modelid,
                        path=r'../GAN models/Spatial models/NIR - based/Temporal')
            print("Save successful")

        g_epoch, d_epoch = 0.0, 0.0

        # Freeze/unfreeze Gs
        for p in Gs.parameters():
            p.requires_grad_(epoch >= freeze_spatial_epochs)

        batch_counter = 1
        for real_seq, labels in dataloader:
            print(f"Epoch: {epoch} | Batch: {batch_counter}")
            batch_counter += 1

            real_seq = real_seq.to(DEVICE).float()  # [B,T,4,64,64]
            labels = labels.to(DEVICE).long()       # [B,64,64]
            B = real_seq.size(0)

            one_hot = F.one_hot(labels, num_classes=num_classes).permute(0,3,1,2).float()

            # -------- Train Ds (spatial) --------
            with torch.no_grad():
                fake_seq = G(one_hot, T, doy_list)

            real_bt = real_seq.view(B*T, 4, 64, 64)
            fake_bt = fake_seq.view(B*T, 4, 64, 64)
            label_bt = one_hot.unsqueeze(1).repeat(1, T, 1, 1, 1).view(B*T, -1, 64, 64)

            real_targets = torch.empty(B*T, 1, device=DEVICE).uniform_(0.9, 1.0)
            fake_targets = torch.empty(B*T, 1, device=DEVICE).uniform_(0.0, 0.1)

            if torch.rand(1).item() < 0.1:
                real_targets, fake_targets = fake_targets, real_targets

            d_real, _ = Ds(real_bt, label_bt)
            d_fake, _ = Ds(fake_bt, label_bt)

            d_loss = loss_bce(d_real, real_targets) + loss_bce(d_fake, fake_targets)
            d_optimizer.zero_grad()
            d_loss.backward()
            d_optimizer.step()

            # -------- Train Dt (temporal) --------
            with torch.no_grad():
                _, real_feats_bt = Ds(real_bt, label_bt)
                _, fake_feats_bt = Ds(fake_bt, label_bt)
                real_feats_bt = torch.cat(real_feats_bt, dim=1)
                fake_feats_bt = torch.cat(fake_feats_bt, dim=1)

            real_feat_seq = real_feats_bt.view(B, T, -1)
            fake_feat_seq = fake_feats_bt.view(B, T, -1)

            real_seq_targets = torch.empty(B, 1, device=DEVICE).uniform_(0.9, 1.0)
            fake_seq_targets = torch.empty(B, 1, device=DEVICE).uniform_(0.0, 0.1)

            dt_real = Dt(real_feat_seq)
            dt_fake = Dt(fake_feat_seq)

            d_loss_t = loss_bce(dt_real, real_seq_targets) + loss_bce(dt_fake, fake_seq_targets)
            d_optimizer_t.zero_grad()
            d_loss_t.backward()
            d_optimizer_t.step()

            # -------- Train G (Gs + Gt) --------
            g_iter_loss_accum = 0.0
            for _ in range(gen_iter):
                fake_seq = G(one_hot, T, doy_list)
                fake_bt = fake_seq.view(B*T, 4, 64, 64)

                g_pred_s, fake_feats_bt = Ds(fake_bt, label_bt)
                g_targets_s = torch.ones(B*T, 1, device=DEVICE)
                adv_spatial = loss_bce(g_pred_s, g_targets_s)

                with torch.no_grad():
                    _, real_feats_bt = Ds(real_bt, label_bt)

                real_feats_bt = torch.cat(real_feats_bt, dim=1)
                fake_feats_bt = torch.cat(fake_feats_bt, dim=1)

                feat_match = F.l1_loss(fake_feats_bt, real_feats_bt)

                fake_feat_seq = fake_feats_bt.view(B, T, -1)
                g_pred_t = Dt(fake_feat_seq)
                g_targets_t = torch.ones(B, 1, device=DEVICE)
                adv_temporal = loss_bce(g_pred_t, g_targets_t)

                recon = F.l1_loss(fake_seq, real_seq)

                # ----- Updated tv_t (time-weighted) -----
                delta_frames = torch.abs(fake_seq[:, 1:] - fake_seq[:, :-1])  # [B, T-1, C, H, W]
                tv_t = torch.mean(weights_tv * delta_frames)  # time-aware smoothness

                # ----- Differentiable DTW loss -----
                dtw_loss = 0.0
                for b in range(B):
                    dtw_loss += dtw_loss_fn(fake_feat_seq[b], real_feat_seq[b])
                dtw_loss = dtw_loss / B

                g_loss = (w_adv_spatial * adv_spatial
                          + w_adv_temporal * adv_temporal
                          + w_featmatch * feat_match
                          + w_recon * recon
                          + w_smooth * tv_t
                          + w_dtw * dtw_loss)

                g_optimizer.zero_grad()
                g_loss.backward()
                g_optimizer.step()
                g_iter_loss_accum += g_loss.item()

            g_epoch += g_iter_loss_accum / gen_iter
            d_epoch += (d_loss.item() + d_loss_t.item())

        avg_g = g_epoch / len(dataloader)
        avg_d = d_epoch / len(dataloader)
        print(f"[Epoch {epoch+1}] D: {avg_d:.4f} | G: {avg_g:.4f}")

        losses_g.append(avg_g)
        losses_d.append(avg_d)

        if ((epoch+1)%1) == 0:
            print("Saving models...")
            save_models(G.fetch_spatial_generator(), 
                        G.fetch_temporal_generator(),
                        Ds, Dt, g_optimizer, d_optimizer, d_optimizer_t,
                        epoch, modelid=modelid,
                        path=r'../GAN models/Spatial models/NIR - based/Temporal')
            print("Done.")

    return losses_g, losses_d
